In [2]:
import os
import gc
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

class GoldPipeline:
    def __init__(self, base_dir_path: str):
        self.base_dir = Path(base_dir_path)
        self.lakehouse_dir = self.base_dir / 'lakehouse'

        self.silver_dir = self.lakehouse_dir / 'silver'
        self.gold_dir = self.lakehouse_dir / 'gold'

        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.current_run_dir = self.gold_dir / self.timestamp
        self._setup_directories()

    def _setup_directories(self):
        self.current_run_dir.mkdir(parents=True, exist_ok=True)
        logging.info(f"Gold Run Directory Created: {self.current_run_dir}")

    def _optimize_memory(self, df):
        for col in df.select_dtypes(include=['float64']).columns:
            df[col] = df[col].astype('float32')
        for col in df.select_dtypes(include=['int64']).columns:
            df[col] = df[col].astype('int32')

        categorical_cols = ['Outlet_Type', 'Outlet_Size', 'Distributor_ID', 'SKU_ID']
        for col in categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')

        return df

    def load_silver_data(self):
        logging.info("Loading datasets from Silver Layer (and falling back to Bronze for POI/Holidays)...")

        # Load Silver Data
        silver_runs = sorted([d for d in self.silver_dir.iterdir() if d.is_dir() and d.name.startswith('20')], reverse=True)
        target_silver_dir = next((d for d in silver_runs if (d / 'transactions_clean.parquet').exists()), None)

        if not target_silver_dir:
            raise FileNotFoundError("Could not find valid Silver directory containing transactions_clean.parquet.")

        logging.info(f"Using Silver directory: {target_silver_dir.name}")
        self.transactions = pd.read_parquet(target_silver_dir / 'transactions_clean.parquet')
        self.outlets = pd.read_parquet(target_silver_dir / 'outlet_clean.parquet')
        self.coordinates = pd.read_parquet(target_silver_dir / 'coordinates_clean.parquet')
        self.seasonality = pd.read_parquet(target_silver_dir / 'seasonality_clean.parquet')

        # Bronze Fallback Setup
        bronze_dir = self.lakehouse_dir / 'bronze'
        bronze_runs = sorted([d for d in bronze_dir.iterdir() if d.is_dir() and d.name.startswith('20')], reverse=True)
        target_bronze_dir = next((d for d in bronze_runs if (d / 'holiday_list.csv').exists()), None)

        # Load Holidays (Fallback to Bronze)
        try:
            self.holidays = pd.read_parquet(target_silver_dir / 'holidays_clean.parquet')
        except FileNotFoundError:
            logging.warning("holidays_clean.parquet not found in Silver. Falling back to bronze holiday_list.csv.")
            self.holidays = pd.read_csv(target_bronze_dir / 'holiday_list.csv')

        # Load POI Data (Fallback to Bronze)
        try:
            self.poi_data = pd.read_parquet(target_silver_dir / 'poi_clean.parquet')
            logging.info("Successfully loaded external POI data from Silver.")
        except FileNotFoundError:
            logging.warning("POI features not found in Silver. Falling back to Bronze external_poi_raw.csv.")

            poi_path = bronze_dir / 'external_poi_raw.csv'

            if poi_path.exists():
                self.poi_data = pd.read_csv(poi_path)
                logging.info(f"Loaded POI data from Bronze root: {poi_path.name}")
            else:
                logging.warning("POI data completely missing. Skipping spatial merge.")
                self.poi_data = None

        if 'Year' in self.transactions.columns and 'Month' in self.transactions.columns:
            self.transactions['Date'] = pd.to_datetime(self.transactions[['Year', 'Month']].assign(day=1))

        logging.info("All datasets loaded successfully.")

    def feature_engineering_time_and_holidays(self):
        logging.info("Preparing Monthly Holiday features...")
        self.holidays['Date'] = pd.to_datetime(self.holidays['Date'])
        self.holidays['Year'] = self.holidays['Date'].dt.year
        self.holidays['Month'] = self.holidays['Date'].dt.month

        self.monthly_holidays = self.holidays.groupby(['Year', 'Month']).size().reset_index(name='Holiday_Count')
        return self.monthly_holidays

    def feature_engineering_latent_demand_and_lags(self):
        logging.info(f"Building time grid and engineering Latent Demand logic...")

        if 'Volume_Liters' in self.transactions.columns:
            self.transactions['Volume_Liters'] = pd.to_numeric(self.transactions['Volume_Liters'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0.0).astype('float32')

        if 'Total_Bill_Value' in self.transactions.columns:
            self.transactions['Total_Bill_Value'] = pd.to_numeric(self.transactions['Total_Bill_Value'].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(0.0).astype('float32')

        agg_dict = {
            'Volume_Liters': 'sum',
            'Total_Bill_Value': 'sum',
            'Distributor_ID': 'first'
        }
        base_txn = self.transactions.groupby(['Outlet_ID', 'SKU_ID', 'Date'], as_index=False).agg(agg_dict)

        # Calculate Outlet-level Monthly Activity
        outlet_monthly_activity = base_txn.groupby(['Outlet_ID', 'Date'])['Total_Bill_Value'].sum().reset_index()
        outlet_monthly_activity.rename(columns={'Total_Bill_Value': 'Outlet_Monthly_Total_Bill'}, inplace=True)
        outlet_monthly_activity['Outlet_Monthly_Total_Bill'] = outlet_monthly_activity['Outlet_Monthly_Total_Bill'].astype('float32')

        min_date, max_date = base_txn['Date'].min(), base_txn['Date'].max()
        all_dates = pd.date_range(start=min_date, end=max_date, freq='MS')
        unique_pairs = base_txn[['Outlet_ID', 'SKU_ID', 'Distributor_ID']].drop_duplicates()

        dates_df = pd.DataFrame({'Date': all_dates})
        grid = pd.merge(unique_pairs.assign(key=1), dates_df.assign(key=1), on='key').drop('key', axis=1)

        del unique_pairs
        del dates_df
        gc.collect()

        self.transactions = pd.merge(grid, base_txn.drop(columns=['Distributor_ID']), on=['Outlet_ID', 'SKU_ID', 'Date'], how='left')

        del base_txn
        del grid
        gc.collect()

        self.transactions = pd.merge(self.transactions, outlet_monthly_activity, on=['Outlet_ID', 'Date'], how='left')

        del outlet_monthly_activity
        gc.collect()

        self.transactions['Year'] = self.transactions['Date'].dt.year.astype('int16')
        self.transactions['Month'] = self.transactions['Date'].dt.month.astype('int8')

        self.transactions = pd.merge(self.transactions, self.monthly_holidays, on=['Year', 'Month'], how='left')
        self.transactions['Holiday_Count'] = self.transactions['Holiday_Count'].fillna(0).astype('int8')

        if 'Month' in self.seasonality.columns and 'Distributor_ID' in self.seasonality.columns:
            season_dedup = self.seasonality.drop_duplicates(subset=['Distributor_ID', 'Month']).copy()

            # Defensively drop metadata columns like 'Year' or 'Date' from seasonality
            cols_to_drop = [c for c in ['Year', 'Date'] if c in season_dedup.columns]
            if cols_to_drop:
                season_dedup = season_dedup.drop(columns=cols_to_drop)

            self.transactions = pd.merge(self.transactions, season_dedup, on=['Distributor_ID', 'Month'], how='left')
        else:
            logging.warning("Seasonality data missing 'Month' or 'Distributor_ID' column.")

        group_cols = ['Outlet_ID', 'SKU_ID']
        self.transactions = self.transactions.sort_values(by=group_cols + ['Date'])

        self.transactions['Is_Missing'] = self.transactions['Volume_Liters'].isna()
        self.transactions['Is_Stockout'] = self.transactions['Is_Missing'] & (self.transactions['Outlet_Monthly_Total_Bill'].fillna(0) > 0)

        self.transactions['Volume_Liters_Imputed'] = self.transactions.groupby(group_cols)['Volume_Liters'].transform(lambda x: x.ffill(limit=1).fillna(0.0)).astype('float32')

        self.transactions['Rolling_6M_Max_Volume'] = self.transactions.groupby(group_cols)['Volume_Liters_Imputed'].transform(
            lambda x: x.rolling(window=6, min_periods=1).max()
        ).astype('float32')

        seasonality_col = 'Seasonality_Index' if 'Seasonality_Index' in self.transactions.columns else None

        self.transactions['Latent_Demand_Proxy'] = np.where(
            self.transactions['Is_Stockout'],
            self.transactions['Rolling_6M_Max_Volume'],
            self.transactions['Volume_Liters_Imputed']
        ).astype('float32')

        if seasonality_col:
            self.transactions[seasonality_col] = pd.to_numeric(self.transactions[seasonality_col].astype(str).str.replace(',', '', regex=False), errors='coerce').fillna(1.0).astype('float32')
            self.transactions['Latent_Demand_Proxy'] = (self.transactions['Latent_Demand_Proxy'] * self.transactions[seasonality_col]).astype('float32')

        # Added .fillna(0.0) so the model doesn't fail on the first 1-2 months of data
        self.transactions['Lag_1M_Volume'] = self.transactions.groupby(group_cols)['Volume_Liters_Imputed'].shift(1).fillna(0.0).astype('float32')
        self.transactions['Lag_2M_Volume'] = self.transactions.groupby(group_cols)['Volume_Liters_Imputed'].shift(2).fillna(0.0).astype('float32')

        self.transactions = self.transactions.drop(columns=['Volume_Liters'])

        self.transactions = self._optimize_memory(self.transactions)
        gc.collect()

    def _calculate_spatial_poi_features(self, outlets_df, poi_df, radius_km=2.0):
        # The outlet master has 'Lat' and 'Lon', but POI data has 'Latitude' and 'Longitude'
        lat1 = np.radians(outlets_df['Lat'].values)[:, np.newaxis]
        lon1 = np.radians(outlets_df['Lon'].values)[:, np.newaxis]
        lat2 = np.radians(poi_df['Latitude'].values)[np.newaxis, :]
        lon2 = np.radians(poi_df['Longitude'].values)[np.newaxis, :]

        a = np.sin((lat2 - lat1) / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0)**2
        distances = 2 * np.arcsin(np.sqrt(a)) * 6371.0

        poi_counts = np.sum(distances <= radius_km, axis=1)

        poi_features_df = pd.DataFrame({
            'Outlet_ID': outlets_df['Outlet_ID'],
            f'POI_Count_Within_{int(radius_km)}km': poi_counts.astype('int16')
        })

        return pd.merge(outlets_df, poi_features_df, on='Outlet_ID', how='left')

    def build_outlet_master_features(self):
        logging.info("Merging static Outlet dimension features...")
        outlets_clean = self.outlets.drop_duplicates(subset=['Outlet_ID'])
        coords_clean = self.coordinates.drop_duplicates(subset=['Outlet_ID'])

        self.outlet_features = pd.merge(outlets_clean, coords_clean, on='Outlet_ID', how='left')

        if self.poi_data is not None:
            self.outlet_features = self._calculate_spatial_poi_features(self.outlet_features, self.poi_data, radius_km=2.0)

        self.outlet_features = self._optimize_memory(self.outlet_features)
        return self.outlet_features

    def run(self):
        logging.info("========== GOLD LAYER STARTED ==========")

        self.load_silver_data()
        self.feature_engineering_time_and_holidays()
        self.feature_engineering_latent_demand_and_lags()
        self.build_outlet_master_features()

        logging.info("Creating Final Master Feature Store...")

        overlapping_cols = [col for col in self.outlet_features.columns if col in self.transactions.columns and col != 'Outlet_ID']
        if overlapping_cols:
            self.outlet_features = self.outlet_features.drop(columns=overlapping_cols)

        self.transactions = self.transactions.merge(self.outlet_features, on='Outlet_ID', how='left')

        del self.outlet_features
        gc.collect()

        self.transactions = self.transactions.drop_duplicates(subset=['Outlet_ID', 'SKU_ID', 'Date'])

        output_path = self.current_run_dir / 'master_feature_store.parquet'
        self.transactions.to_parquet(output_path, index=False, engine='pyarrow')

        logging.info(f"SUCCESS -> Master feature store saved at {output_path}")
        logging.info(f"Final Clean Dataset Shape: {self.transactions.shape}")
        logging.info("========== GOLD LAYER COMPLETED ==========")
        return self.transactions

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s', force=True)

    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists(): drive.mount('/content/drive', force_remount=True)
    except ImportError:
        pass

    BASE_DIR = '/content/drive/MyDrive/DataStorm_TeamZypher'

    try:
        gold_pipeline = GoldPipeline(base_dir_path=BASE_DIR)
        final_ml_dataset = gold_pipeline.run()
        print(final_ml_dataset.head())
    except Exception as e:
        logging.error(f"Gold Layer failed: {str(e)}")

2026-05-16 23:36:46,995 - [INFO] - Gold Run Directory Created: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/gold/20260516_233646
2026-05-16 23:36:47,005 - [INFO] - ========== GOLD LAYER STARTED ==========
2026-05-16 23:36:47,006 - [INFO] - Loading datasets from Silver Layer (and falling back to Bronze for POI/Holidays)...
2026-05-16 23:36:47,020 - [INFO] - Using Silver directory: 20260515_185133
2026-05-16 23:36:48,846 - [WARNING] - holidays_clean.parquet not found in Silver. Falling back to bronze holiday_list.csv.
2026-05-16 23:36:48,857 - [WARNING] - POI features not found in Silver. Falling back to Bronze external_poi_raw.csv.
2026-05-16 23:36:48,883 - [INFO] - Loaded POI data from Bronze root: external_poi_raw.csv
2026-05-16 23:36:49,046 - [INFO] - All datasets loaded successfully.
2026-05-16 23:36:49,047 - [INFO] - Preparing Monthly Holiday features...
2026-05-16 23:36:49,057 - [INFO] - Building time grid and engineering Latent Demand logic...
2026-05-16 23:39:13,979 - [

   Outlet_ID  SKU_ID Distributor_ID       Date  Total_Bill_Value  \
0  OUT_00001  SKU_01      DIST_W_03 2023-01-01               NaN   
1  OUT_00001  SKU_01      DIST_W_03 2023-02-01               NaN   
2  OUT_00001  SKU_01      DIST_W_03 2023-03-01      19829.292969   
3  OUT_00001  SKU_01      DIST_W_03 2023-04-01               NaN   
4  OUT_00001  SKU_01      DIST_W_03 2023-05-01               NaN   

   Outlet_Monthly_Total_Bill  Year  Month  Holiday_Count  Seasonality_Index  \
0                        NaN  2023      1             20                1.0   
1                        NaN  2023      2             20                1.0   
2                 398271.125  2023      3              8                1.0   
3                        NaN  2023      4             26                1.0   
4                        NaN  2023      5             26                1.0   

   ...  Rolling_6M_Max_Volume  Latent_Demand_Proxy  Lag_1M_Volume  \
0  ...               0.000000             0.000